# Stage 0 Evaluation
Tests each module — `cleaner.py`, `splitter.py`, `pipeline.py` — on real case files.

In [45]:
# ── Setup: fix path so imports work when notebook runs from stage0/ ──────────
import sys, os

# Make sure the project root (parent of stage0/) is on sys.path
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from stage0.cleaner  import clean_text
from stage0.splitter import split_sentences
from stage0.pipeline import process_case, save_stage0_output, process_all_cases

CASES_DIR  = os.path.join(ROOT, "data", "cases")
STAGE0_DIR = os.path.join(ROOT, "data", "stage0")
SAMPLE     = os.path.join(CASES_DIR, "C14.txt")   # change to any case

print("Root:", ROOT)
print("Sample file exists:", os.path.exists(SAMPLE))

Root: d:\Project
Sample file exists: True


---
## 1 · Raw text preview

In [46]:
# Read and show the first 60 lines of the raw file
with open(SAMPLE, encoding="utf-8", errors="replace") as f:
    raw_lines = f.readlines()

print(f"Total lines in raw file: {len(raw_lines)}")
print("\n── First 60 lines ──────────────────────────────────────")
print("".join(raw_lines[:60]))

Total lines in raw file: 326

── First 60 lines ──────────────────────────────────────
Central Inland Water Transport Corporation Limited and Another v Brojo Nath Ganguly and Another
Supreme Court of India

6 April 1986
C.A. No. 4412 and 4413 of 1985
The Judgment was delivered by : D. P. Madon, J.
1. These Appeals by Special Leave granted by this Court raise two questions of considerable importance to Government companies and their employees including their officers. These questions are:
1) Whether a Government company as defined in s. 617 of the Companies Act, 1956, is "the State" within the meaning of Art. 12 of the Constitution?
2) Whether an unconscionable term in a contract of employment is void u/s. 23 of the Indian Contract Act, 1872, as being opposed to public policy and, when such a term is contained in a contract of employment entered into with a Government company, is also void as infringing Art. 14 of the Constitution in case a Government company is "the State" under Art. 1

---
## 2 · cleaner.py — `clean_text()`

In [47]:
raw_text = "".join(raw_lines)
cleaned  = clean_text(raw_text)

raw_chars     = len(raw_text)
cleaned_chars = len(cleaned)
reduction_pct = (1 - cleaned_chars / raw_chars) * 100

print(f"Raw chars   : {raw_chars:,}")
print(f"Cleaned chars: {cleaned_chars:,}")
print(f"Reduction   : {reduction_pct:.1f}%")
print("\n── Cleaned text (first 2 000 chars) ────────────────────")
print(cleaned[:2000])

Raw chars   : 196,084
Cleaned chars: 173,431
Reduction   : 11.6%

── Cleaned text (first 2 000 chars) ────────────────────
Central Inland Water Transport Corporation Limited and Another v Brojo Nath Ganguly and Another

1. These Appeals by Special Leave granted by this Court raise two questions of considerable importance to Government companies and their employees including their officers. These questions are:
1) Whether a Government company as defined in s. 617 of the Companies Act, 1956, is "the State" within the meaning of Art. 12 of the Constitution?
2) Whether an unconscionable term in a contract of employment is void u/s. 23 of the Indian Contract Act, 1872, as being opposed to public policy and, when such a term is contained in a contract of employment entered into with a Government company, is also void as infringing Art. 14 of the Constitution in case a Government company is "the State" under Art. 12 of the Constitution?
2. Although the record of these Appeals is voluminous, t

In [48]:
# Side-by-side: show lines that were REMOVED by the cleaner
raw_line_set     = set(l.strip() for l in raw_lines if l.strip())
cleaned_line_set = set(cleaned.splitlines())
removed_lines    = [l for l in raw_line_set if l not in cleaned_line_set]

print(f"Lines removed by cleaner: {len(removed_lines)}")
print("\nSample removed lines (first 20):")
for line in removed_lines[:20]:
    print(" >", line[:120])

Lines removed by cleaner: 50

Sample removed lines (first 20):
 > 97. He then referred to various categories of cases and ultimately deduced therefrom a general principle in these words 
 > 109. There are two schools of thought - "the narrow view" school and "the broad view" school. According to the former, c
 > 127. We must now turn to two decisions of the Bombay High Court as each party has relied strongly upon one of them, name
 > What the French call "contracts d'adhesion', the American call A "adhesion contracts" or "contracts of adhesion." An "ad
 > C.A. No. 4412 and 4413 of 1985
 > Supreme Court of India
 > 113. We will now test the validity of Rule 9(i) by applying to it the principle formulated above. Each of the contesting
 > Appeal dismissed.
 > 124. As the Corporation is "the State" within the meaning of Article 12, it was amenable to the writ jurisdiction of the
 > 99. Lord Diplock then proceeded to point out that there are two kinds of standard forms of contracts. The fir

In [49]:
# Idempotence check: clean_text(clean_text(x)) == clean_text(x)
double_cleaned = clean_text(cleaned)
assert double_cleaned == cleaned, "Idempotence FAILED"
print("Idempotence check: PASSED ✓")

Idempotence check: PASSED ✓


---
## 3 · splitter.py — `split_sentences()`

In [50]:
sentences = split_sentences(cleaned)

print(f"Total sentences: {len(sentences)}")
word_counts = [len(s.split()) for s in sentences]
print(f"Avg words/sentence : {sum(word_counts)/len(word_counts):.1f}")
print(f"Min words/sentence : {min(word_counts)}")
print(f"Max words/sentence : {max(word_counts)}")

print("\n── First 10 sentences ──────────────────────────────────")
for i, s in enumerate(sentences[:10], 1):
    print(f"[{i:02d}] {s[:150]}")

Total sentences: 852
Avg words/sentence : 34.4
Min words/sentence : 5
Max words/sentence : 198

── First 10 sentences ──────────────────────────────────
[01] Central Inland Water Transport Corporation Limited and Another v Brojo Nath Ganguly and Another 1.
[02] These Appeals by Special Leave granted by this Court raise two questions of considerable importance to Government companies and their employees includ
[03] These questions are:
1) Whether a Government company as defined in s. 617 of the Companies Act, 1956, is "the State" within the meaning of Art. 12 of 
[04] Although the record of these Appeals is voluminous, the salient facts lie within a narrow compass.
[05] The First Appellant in both these Appeals, namely, the Central Inland Water Transport Corporation Limited (hereinafter referred to in short as "the Co
[06] The majority of the shares of the Corporation were at all times and still are held by the Union of India which is the Second Respondent in these Appea
[07] S. 617 of 

In [51]:
# Quality checks
empty_sents   = [s for s in sentences if not s.strip()]
short_sents   = [s for s in sentences if len(s.split()) < 5]
leading_space = [s for s in sentences if s != s.strip()]

print(f"Empty sentences      : {len(empty_sents)}  (should be 0)")
print(f"Sentences < 5 words  : {len(short_sents)}  (should be 0)")
print(f"Unstripped sentences : {len(leading_space)}  (should be 0)")

assert len(empty_sents)   == 0, "Found empty sentences!"
assert len(short_sents)   == 0, "Found sentences with < 5 words!"
assert len(leading_space) == 0, "Found unstripped sentences!"
print("\nAll splitter checks: PASSED ✓")

Empty sentences      : 0  (should be 0)
Sentences < 5 words  : 0  (should be 0)
Unstripped sentences : 0  (should be 0)

All splitter checks: PASSED ✓


---
## 4 · pipeline.py — `process_case()` + `save_stage0_output()`

In [52]:
import json

case_id   = os.path.splitext(os.path.basename(SAMPLE))[0]  # "C14"
sentences = process_case(SAMPLE)

print(f"case_id  : {case_id}")
print(f"Sentences: {len(sentences)}")

save_stage0_output(case_id, sentences, output_dir=STAGE0_DIR)

out_path = os.path.join(STAGE0_DIR, f"{case_id}.json")
print(f"\nSaved → {out_path}")

# Round-trip check
with open(out_path, encoding="utf-8") as f:
    loaded = json.load(f)

assert loaded["case_id"]   == case_id,   "case_id mismatch!"
assert loaded["sentences"] == sentences, "sentences mismatch!"
print("Round-trip JSON check: PASSED ✓")

print("\n── First 5 sentences from JSON ─────────────────────────")
for i, s in enumerate(loaded["sentences"][:5], 1):
    print(f"[{i}] {s[:150]}")

case_id  : C14
Sentences: 852

Saved → d:\Project\data\stage0\C14.json
Round-trip JSON check: PASSED ✓

── First 5 sentences from JSON ─────────────────────────
[1] Central Inland Water Transport Corporation Limited and Another v Brojo Nath Ganguly and Another 1.
[2] These Appeals by Special Leave granted by this Court raise two questions of considerable importance to Government companies and their employees includ
[3] These questions are:
1) Whether a Government company as defined in s. 617 of the Companies Act, 1956, is "the State" within the meaning of Art. 12 of 
[4] Although the record of these Appeals is voluminous, the salient facts lie within a narrow compass.
[5] The First Appellant in both these Appeals, namely, the Central Inland Water Transport Corporation Limited (hereinafter referred to in short as "the Co


---
## 5 · Compare raw vs cleaned vs split (visual diff on one paragraph)

In [53]:
# Pick the first 10 raw lines as a mini-sample to trace through the pipeline
sample_raw     = "".join(raw_lines[:10])
sample_cleaned = clean_text(sample_raw)
sample_sents   = split_sentences(sample_cleaned)

print("── RAW ─────────────────────────────────────────────────")
print(sample_raw)
print("── AFTER clean_text() ──────────────────────────────────")
print(sample_cleaned)
print("── AFTER split_sentences() ─────────────────────────────")
for i, s in enumerate(sample_sents, 1):
    print(f"  [{i}] {s}")

── RAW ─────────────────────────────────────────────────
Central Inland Water Transport Corporation Limited and Another v Brojo Nath Ganguly and Another
Supreme Court of India

6 April 1986
C.A. No. 4412 and 4413 of 1985
The Judgment was delivered by : D. P. Madon, J.
1. These Appeals by Special Leave granted by this Court raise two questions of considerable importance to Government companies and their employees including their officers. These questions are:
1) Whether a Government company as defined in s. 617 of the Companies Act, 1956, is "the State" within the meaning of Art. 12 of the Constitution?
2) Whether an unconscionable term in a contract of employment is void u/s. 23 of the Indian Contract Act, 1872, as being opposed to public policy and, when such a term is contained in a contract of employment entered into with a Government company, is also void as infringing Art. 14 of the Constitution in case a Government company is "the State" under Art. 12 of the Constitution?
2. Alth

---
## 6 · Batch: run on first N cases

In [54]:
# Process a small batch (first 10 files) to verify batch pipeline
import glob

all_txt = sorted(glob.glob(os.path.join(CASES_DIR, "*.txt")))
batch   = all_txt[:10]

batch_results = {}
for fp in batch:
    cid  = os.path.splitext(os.path.basename(fp))[0]
    sents = process_case(fp)
    save_stage0_output(cid, sents, output_dir=STAGE0_DIR)
    batch_results[cid] = sents
    print(f"  {cid}: {len(sents)} sentences")

print(f"\nBatch done. {len(batch_results)} files processed.")

  C1: 37 sentences
  C10: 198 sentences
  C100: 190 sentences
  C1000: 79 sentences
  C1001: 19 sentences
  C1002: 113 sentences
  C1003: 96 sentences
  C1004: 130 sentences
  C1005: 53 sentences
  C1006: 177 sentences

Batch done. 10 files processed.


In [55]:
# Summary stats across the batch
counts = {cid: len(sents) for cid, sents in batch_results.items()}
total  = sum(counts.values())
avg    = total / len(counts)

print(f"{'Case':<8} {'Sentences':>10}")
print("-" * 20)
for cid, n in sorted(counts.items()):
    print(f"{cid:<8} {n:>10}")
print("-" * 20)
print(f"{'Total':<8} {total:>10}")
print(f"{'Avg':<8} {avg:>10.1f}")

Case      Sentences
--------------------
C1               37
C10             198
C100            190
C1000            79
C1001            19
C1002           113
C1003            96
C1004           130
C1005            53
C1006           177
--------------------
Total          1092
Avg           109.2
